# MCP client

> Local or remote MCP servers, named in a project `.agents/mcp.json` or `<cfg>/mcp.json`, wrapped
> as ordinary Ramabana tools with [AnswerDotAI/mcpmini](https://github.com/AnswerDotAI/mcpmini) --
> the mirror of `ramabana.mcp`, which serves tools rather than consuming them.

In [ ]:
#| default_exp mcpclient

In [ ]:
#| export
import asyncio, json, os, threading
from pathlib import Path

from mcpmini.core import MCPClient
from shalya.core import writes

DFLT_MCP_TIMEOUT = 60   #: seconds one MCP call may take before the caller gives up on it


In [ ]:
#| export
class MCPLoop:
    "One background asyncio loop, shared by every MCP client a session opens: mcpmini is async, Ramabana's tools are not."

    def __init__(self):
        self.loop = asyncio.new_event_loop()
        self.thread = threading.Thread(target=self.loop.run_forever, daemon=True, name='ramabana-mcp-client')
        self.thread.start()

    def run(self, coro, timeout=DFLT_MCP_TIMEOUT):
        "Run `coro` to completion on the background loop and return its result."
        return asyncio.run_coroutine_threadsafe(coro, self.loop).result(timeout)

    def close(self):
        "Stop the loop; its thread ends once whatever call is in flight returns."
        self.loop.call_soon_threadsafe(self.loop.stop)


In [ ]:
#| export
def mcp_config(cfg, roots, extra=None, project=False):
    "`{name: spec}` for every server: `extra` (an editor's own `mcp_servers`) first, then a project `.agents/mcp.json` (only when `project` opts in, matching extensions), then `<cfg>/mcp.json`. Earlier entries win by name."
    found = dict(extra or {})
    paths = [Path(r)/'.agents'/'mcp.json' for r in roots or ()] if project else []
    if cfg is not None: paths.append(Path(cfg)/'mcp.json')
    for p in paths:
        if not p.is_file(): continue
        try: servers = json.loads(p.read_text()).get('servers', {})
        except Exception: continue
        for name, spec in servers.items(): found.setdefault(name, spec)
    return found


In [ ]:
#| export
def _stdio_env(env):
    "The child's environment: this process's own, plus `env`'s overrides. `mcpmini` only inherits when `env` is None."
    return {**os.environ, **env} if env else None

def _tool_hints(raw):
    "`{name: annotations}` from a `tools/list` reply; a missing or malformed `annotations` becomes `{}`, never a crash."
    return {t['name']: (a if isinstance(a := t.get('annotations'), dict) else {}) for t in raw}

def _connect(spec, loop):
    "A started `MCPClient` for `spec` (`{'command': [...], 'env': {...}}` or `{'url': ..., 'token': ...}`), and its tools' annotations by name."
    client = (MCPClient.http(spec['url'], token=spec.get('token')) if 'url' in spec else
             MCPClient.stdio(spec['command'], env=_stdio_env(spec.get('env'))))
    loop.run(client.start())
    raw = loop.run(client.rpc('tools/list'))['tools']
    return client, _tool_hints(raw)

def _wrap(client, name, hints, loop):
    "One MCP tool as a plain Python callable, marked a write unless the server declared it read-only."
    fn = client.tools[name]
    def call(*a, **kw): return loop.run(fn(*a, **kw))
    call.__name__ = call.__qualname__ = fn.__name__
    call.__doc__, call.__signature__, call.__annotations__ = fn.__doc__, fn.__signature__, fn.__annotations__
    return call if hints.get(name, {}).get('readOnlyHint') is True else writes(call)


In [ ]:
#| export
def mcp_tools(cfg, roots, extra=None, project=False, loop=None, on_error=None):
    "Every tool an `mcp_config` server offers, as plain Python callables; the loop they run on; the connected clients, so a caller can close them. A server that cannot connect is skipped, not fatal."
    servers = mcp_config(cfg, roots, extra, project)
    if not servers: return [], loop, []
    loop = loop or MCPLoop()
    out, clients = [], []
    for name, spec in servers.items():
        try: client, hints = _connect(spec, loop)
        except Exception as e:
            if on_error is not None: on_error(name, e)
            continue
        clients.append(client)
        out += [_wrap(client, tool_name, hints, loop) for tool_name in client.tools]
    return out, loop, clients

def close_mcp_clients(clients, loop):
    "Close every connected client's transport on `loop`, before the loop itself stops."
    for client in clients:
        try: loop.run(client.tr.aclose())
        except Exception: pass


In [ ]:
#| export
def acp_mcp_spec(server):
    "`(name, spec)` in `mcp_config` shape for one ACP `mcp_servers` entry (stdio, http or sse)."
    if hasattr(server, 'command'):
        env = {e.name: e.value for e in (getattr(server, 'env', None) or [])}
        return server.name, {'command': [server.command, *(getattr(server, 'args', None) or [])], 'env': env}
    headers = {h.name: h.value for h in (getattr(server, 'headers', None) or [])}
    token = next((v[7:] for k, v in headers.items() if k.lower() == 'authorization' and v.startswith('Bearer ')), None)
    return server.name, {'url': server.url, 'token': token}
